# Optional Lab: Gradient Descent for Linear Regression

<figure>
    <center> <img src="../work/images/C1_W1_L4_S1_Lecture_GD.png"  style="width:800px;height:200px;" ></center>
</figure>

# 可选实验：线性回归的梯度下降（Gradient Descent）

## Goals
In this lab, you will:
- automate the process of optimizing $w$ and $b$ using gradient descent.

## 目标
在本实验中，你将：
- 使用梯度下降自动优化参数 $w$ 和 $b$。

## Tools
In this lab, we will make use of: 
- NumPy, a popular library for scientific computing
- Matplotlib, a popular library for plotting data
- plotting routines in the lab_utils.py file in the local directory

## 工具
在本实验中，我们将使用：
- NumPy：一个流行的科学计算库
- Matplotlib：一个流行的数据绘图库
- 本地目录中 `lab_utils.py` 文件里的绘图例程

In [1]:
import math, copy
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')
from lab_utils_uni import plt_house_x, plt_contour_wgrad, plt_divergence, plt_gradients

ModuleNotFoundError: No module named 'lab_utils_uni'

<a name="toc_40291_2"></a>
# Problem Statement

Let's use the same two data points as before - a house with 1000 square feet sold for \\$300,000 and a house with 2000 square feet sold for \\$500,000.

| Size (1000 sqft)     | Price (1000s of dollars) |
| ----------------| ------------------------ |
| 1               | 300                      |
| 2               | 500                      |


# 问题描述

让我们继续使用与之前相同的两个数据点——一套 1000 平方英尺的房屋售价为 300,000 美元，一套 2000 平方英尺的房屋售价为 500,000 美元。

| 面积（1000 平方英尺） | 价格（千美元） |
| ----------------| ------------------------ |
| 1 | 300 |
| 2 | 500 |

In [ ]:
# Load our data set
x_train = np.array([1.0, 2.0])   #features
y_train = np.array([300.0, 500.0])   #target value

<a name="toc_40291_2.0.1"></a>
### Compute_Cost
This was developed in the last lab. We'll need it again here.

### Compute_Cost
该函数已在上一个实验中编写完成，我们在这里会再次用到它。

In [ ]:
#Function to calculate the cost
def compute_cost(x, y, w, b):
   
    m = x.shape[0] 
    cost = 0
    
    for i in range(m):
        f_wb = w * x[i] + b
        cost = cost + (f_wb - y[i])**2
    total_cost = 1 / (2 * m) * cost

    return total_cost

<a name="toc_40291_2.1"></a>
## Gradient descent summary
So far in this course, you have developed a linear model that predicts $f_{w,b}(x^{(i)})$:
$$f_{w,b}(x^{(i)}) = wx^{(i)} + b \tag{1}$$
In linear regression, you utilize input training data to fit the parameters $w$,$b$ by minimizing a measure of the error between our predictions $f_{w,b}(x^{(i)})$ and the actual data $y^{(i)}$. The measure is called the $cost$, $J(w,b)$. In training you measure the cost over all of our training samples $x^{(i)},y^{(i)}$
$$J(w,b) = \frac{1}{2m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)})^2\tag{2}$$ 

## 梯度下降总结
到目前为止，你已经在本课程中构建了一个预测 $f_{w,b}(x^{(i)})$ 的线性模型：
$$f_{w,b}(x^{(i)}) = wx^{(i)} + b \tag{1}$$
在线性回归中，你利用输入的训练数据，通过最小化预测值 $f_{w,b}(x^{(i)})$ 与实际数据 $y^{(i)}$ 之间的误差度量来拟合参数 $w$ 和 $b$。这种度量称为*代价（cost）*，记作 $J(w,b)$。在训练过程中，你会在所有训练样本 $x^{(i)},y^{(i)}$ 上计算代价：
$$J(w,b) = \frac{1}{2m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)})^2\tag{2}$$


In lecture, *gradient descent* was described as:

$$\begin{align*} \text{repeat}&\text{ until convergence:} \; \lbrace \newline
\;  w &= w -  \alpha \frac{\partial J(w,b)}{\partial w} \tag{3}  \; \newline 
 b &= b -  \alpha \frac{\partial J(w,b)}{\partial b}  \newline \rbrace
\end{align*}$$
where, parameters $w$, $b$ are updated simultaneously.  
The gradient is defined as:
$$
\begin{align}
\frac{\partial J(w,b)}{\partial w}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)})x^{(i)} \tag{4}\\
  \frac{\partial J(w,b)}{\partial b}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)}) \tag{5}\\
\end{align}
$$

Here *simultaniously* means that you calculate the partial derivatives for all the parameters before updating any of the parameters.

在课程中，*梯度下降（gradient descent）*被描述为：

$$\begin{align*} \text{repeat}&\text{ until convergence:} \; \lbrace \newline
\;  w &= w -  \alpha \frac{\partial J(w,b)}{\partial w} \tag{3}  \; \newline 
 b &= b -  \alpha \frac{\partial J(w,b)}{\partial b}  \newline \rbrace
\end{align*}$$
其中，参数 $w$ 和 $b$ 是同时更新的。
梯度定义为：
$$
\begin{align}
\frac{\partial J(w,b)}{\partial w}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)})x^{(i)} \tag{4}\\
  \frac{\partial J(w,b)}{\partial b}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)}) \tag{5}\\
\end{align}
$$

这里的*同时（simultaneously）*意味着在更新任何参数之前，先计算出所有参数的偏导数。

<a name="toc_40291_2.2"></a>
## Implement Gradient Descent
You will implement gradient descent algorithm for one feature. You will need three functions. 
- `compute_gradient` implementing equation (4) and (5) above
- `compute_cost` implementing equation (2) above (code from previous lab)
- `gradient_descent`, utilizing compute_gradient and compute_cost

Conventions:
- The naming of python variables containing partial derivatives follows this pattern,$\frac{\partial J(w,b)}{\partial b}$  will be `dj_db`.
- w.r.t is With Respect To, as in partial derivative of $J(wb)$ With Respect To $b$.


## 实现梯度下降
你将为单特征情况实现梯度下降算法。你需要三个函数：
- `compute_gradient`：实现上面的公式 (4) 和 (5)
- `compute_cost`：实现上面的公式 (2)（来自上一个实验的代码）
- `gradient_descent`：调用 `compute_gradient` 和 `compute_cost`

命名约定：
- 存储偏导数的 Python 变量命名遵循如下模式：$\frac{\partial J(w,b)}{\partial b}$ 写作 `dj_db`。
- `w.r.t` 是“With Respect To（关于）”的缩写，例如 $J(w,b)$ 关于 $b$ 的偏导数。

<a name="toc_40291_2.3"></a>
### compute_gradient
<a name='ex-01'></a>
`compute_gradient`  implements (4) and (5) above and returns $\frac{\partial J(w,b)}{\partial w}$,$\frac{\partial J(w,b)}{\partial b}$. The embedded comments describe the operations.

### compute_gradient
`compute_gradient` 实现了上面的公式 (4) 和 (5)，并返回 $\frac{\partial J(w,b)}{\partial w}$ 和 $\frac{\partial J(w,b)}{\partial b}$。代码中的注释详细说明了具体操作。

In [ ]:
def compute_gradient(x, y, w, b): 
    """
    Computes the gradient for linear regression 
    Args:
      x (ndarray (m,)): Data, m examples 
      y (ndarray (m,)): target values
      w,b (scalar)    : model parameters  
    Returns
      dj_dw (scalar): The gradient of the cost w.r.t. the parameters w
      dj_db (scalar): The gradient of the cost w.r.t. the parameter b     
     """
    
    # Number of training examples
    m = x.shape[0]    
    dj_dw = 0
    dj_db = 0
    
    for i in range(m):  
        f_wb = w * x[i] + b 
        dj_dw_i = (f_wb - y[i]) * x[i] 
        dj_db_i = f_wb - y[i] 
        dj_db += dj_db_i
        dj_dw += dj_dw_i 
    dj_dw = dj_dw / m 
    dj_db = dj_db / m 
        
    return dj_dw, dj_db

<br/>

<img align="left" src="../work/images/C1_W1_Lab03_lecture_slopes.PNG"   style="width:340px;" > The lectures described how gradient descent utilizes the partial derivative of the cost with respect to a parameter at a point to update that parameter.   
Let's use our `compute_gradient` function to find and plot some partial derivatives of our cost function relative to one of the parameters, $w_0$.


课程中讲解了梯度下降如何利用代价函数在某一点处关于参数的偏导数来更新该参数。
让我们使用 `compute_gradient` 函数，求取并绘制代价函数关于其中一个参数 $w_0$ 的若干偏导数。

In [ ]:
plt_gradients(x_train,y_train, compute_cost, compute_gradient)
plt.show()

Above, the left plot shows $\frac{\partial J(w,b)}{\partial w}$ or the slope of the cost curve relative to $w$ at three points. On the right side of the plot, the derivative is positive, while on the left it is negative. Due to the 'bowl shape', the derivatives will always lead gradient descent toward the bottom where the gradient is zero.
 
The left plot has fixed $b=100$. Gradient descent will utilize both $\frac{\partial J(w,b)}{\partial w}$ and $\frac{\partial J(w,b)}{\partial b}$ to update parameters. The 'quiver plot' on the right provides a means of viewing the gradient of both parameters. The arrow sizes reflect the magnitude of the gradient at that point. The direction and slope of the arrow reflects the ratio of $\frac{\partial J(w,b)}{\partial w}$ and $\frac{\partial J(w,b)}{\partial b}$ at that point.
Note that the gradient points *away* from the minimum. Review equation (3) above. The scaled gradient is *subtracted* from the current value of $w$ or $b$. This moves the parameter in a direction that will reduce cost.

上面左图展示了三个点处代价曲线关于 $w$ 的斜率，即 $\frac{\partial J(w,b)}{\partial w}$。在图的右侧，导数为正；在左侧，导数为负。由于呈“碗状”，导数总会引导梯度下降朝向梯度为零的底部移动。

左图固定了 $b=100$。而梯度下降会同时利用 $\frac{\partial J(w,b)}{\partial w}$ 和 $\frac{\partial J(w,b)}{\partial b}$ 来更新参数。右侧的“箭头图（quiver plot）”提供了一种同时观察两个参数梯度的方法。箭头大小反映了该点处梯度的幅值，箭头的方向和斜率反映了该点处 $\frac{\partial J(w,b)}{\partial w}$ 与 $\frac{\partial J(w,b)}{\partial b}$ 的比值。
注意，梯度方向指向*远离*最小值的方向。回顾上面的公式 (3)：我们需要从当前的 $w$ 或 $b$ 中*减去*缩放后的梯度，从而使参数朝着减小代价的方向移动。

<a name="toc_40291_2.5"></a>
###  Gradient Descent
Now that gradients can be computed,  gradient descent, described in equation (3) above can be implemented below in `gradient_descent`. The details of the implementation are described in the comments. Below, you will utilize this function to find optimal values of $w$ and $b$ on the training data.

### 梯度下降
既然已经能够计算梯度，现在就可以在下面的 `gradient_descent` 函数中实现公式 (3) 所描述的梯度下降算法。实现细节已在注释中说明。接下来，你将使用该函数在训练数据上寻找 $w$ 和 $b$ 的最优值。

In [ ]:
def gradient_descent(x, y, w_in, b_in, alpha, num_iters, cost_function, gradient_function): 
    """
    Performs gradient descent to fit w,b. Updates w,b by taking 
    num_iters gradient steps with learning rate alpha
    
    Args:
      x (ndarray (m,))  : Data, m examples 
      y (ndarray (m,))  : target values
      w_in,b_in (scalar): initial values of model parameters  
      alpha (float):     Learning rate
      num_iters (int):   number of iterations to run gradient descent
      cost_function:     function to call to produce cost
      gradient_function: function to call to produce gradient
      
    Returns:
      w (scalar): Updated value of parameter after running gradient descent
      b (scalar): Updated value of parameter after running gradient descent
      J_history (List): History of cost values
      p_history (list): History of parameters [w,b] 
      """
    
    w = copy.deepcopy(w_in) # avoid modifying global w_in
    # An array to store cost J and w's at each iteration primarily for graphing later
    J_history = []
    p_history = []
    b = b_in
    w = w_in
    
    for i in range(num_iters):
        # Calculate the gradient and update the parameters using gradient_function
        dj_dw, dj_db = gradient_function(x, y, w , b)     

        # Update Parameters using equation (3) above
        b = b - alpha * dj_db                            
        w = w - alpha * dj_dw                            

        # Save cost J at each iteration
        if i<100000:      # prevent resource exhaustion 
            J_history.append( cost_function(x, y, w , b))
            p_history.append([w,b])
        # Print cost every at intervals 10 times or as many iterations if < 10
        if i% math.ceil(num_iters/10) == 0:
            print(f"Iteration {i:4}: Cost {J_history[-1]:0.2e} ",
                  f"dj_dw: {dj_dw: 0.3e}, dj_db: {dj_db: 0.3e}  ",
                  f"w: {w: 0.3e}, b:{b: 0.5e}")
 
    return w, b, J_history, p_history #return w and J,w history for graphing

In [ ]:
# initialize parameters
w_init = 0
b_init = 0
# some gradient descent settings
iterations = 10000
tmp_alpha = 1.0e-2
# run gradient descent
w_final, b_final, J_hist, p_hist = gradient_descent(x_train ,y_train, w_init, b_init, tmp_alpha, 
                                                    iterations, compute_cost, compute_gradient)
print(f"(w,b) found by gradient descent: ({w_final:8.4f},{b_final:8.4f})")

<img align="left" src="../work/images/C1_W1_Lab03_lecture_learningrate.PNG"  style="width:340px; padding: 15px; " > 
Take a moment and note some characteristics of the gradient descent process printed above.  

- The cost starts large and rapidly declines as described in the slide from the lecture.
- The partial derivatives, `dj_dw`, and `dj_db` also get smaller, rapidly at first and then more slowly. As shown in the diagram from the lecture, as the process nears the 'bottom of the bowl' progress is slower due to the smaller value of the derivative at that point.
- progress slows though the learning rate, alpha, remains fixed

请花一点时间观察上面打印出的梯度下降过程的一些特征：

- 如课程幻灯片所述，代价初始值很大，随后迅速下降。
- 偏导数 `dj_dw` 和 `dj_db` 也逐渐变小，起初下降很快，随后变慢。如课程图示所示，随着迭代接近“碗底”，该点处的导数值变小，因此下降速度变慢。
- 尽管学习率 $\alpha$ 保持不变，但每一步的更新幅度会逐渐减小。

### Cost versus iterations of gradient descent 
A plot of cost versus iterations is a useful measure of progress in gradient descent. Cost should always decrease in successful runs. The change in cost is so rapid initially, it is useful to plot the initial decent on a different scale than the final descent. In the plots below, note the scale of cost on the axes and the iteration step.

### 代价随梯度下降迭代次数的变化
代价随迭代次数变化的曲线是衡量梯度下降进展的有效工具。在成功运行的梯度下降中，代价应当始终递减。由于初期代价变化非常迅速，将初始下降阶段与后期下降阶段分别以不同尺度绘图会更清晰。在下面的图表中，请注意坐标轴上代价的刻度以及迭代步数。

In [ ]:
# plot cost versus iteration  
fig, (ax1, ax2) = plt.subplots(1, 2, constrained_layout=True, figsize=(12,4))
ax1.plot(J_hist[:100])
ax2.plot(1000 + np.arange(len(J_hist[1000:])), J_hist[1000:])
ax1.set_title("Cost vs. iteration(start)");  ax2.set_title("Cost vs. iteration (end)")
ax1.set_ylabel('Cost')            ;  ax2.set_ylabel('Cost') 
ax1.set_xlabel('iteration step')  ;  ax2.set_xlabel('iteration step') 
plt.show()

### Predictions
Now that you have discovered the optimal values for the parameters $w$ and $b$, you can now use the model to predict housing values based on our learned parameters. As expected, the predicted values are nearly the same as the training values for the same housing. Further, the value not in the prediction is in line with the expected value.

### 预测
既然你已经找到了参数 $w$ 和 $b$ 的最优值，现在就可以使用模型基于学到的参数预测房价。正如预期的那样，对于训练集中的相同房屋，预测值几乎与训练目标值一致；此外，对于未在训练集中的房屋，预测值也符合预期。

In [ ]:
print(f"1000 sqft house prediction {w_final*1.0 + b_final:0.1f} Thousand dollars")
print(f"1200 sqft house prediction {w_final*1.2 + b_final:0.1f} Thousand dollars")
print(f"2000 sqft house prediction {w_final*2.0 + b_final:0.1f} Thousand dollars")

<a name="toc_40291_2.6"></a>
## Plotting
You can show the progress of gradient descent during its execution by plotting the cost over iterations on a contour plot of the cost(w,b). 

## 绘图
你可以通过在 `cost(w,b)` 的等高线图上绘制迭代过程中的代价变化，来直观展示梯度下降的执行进度。

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(12, 6))
plt_contour_wgrad(x_train, y_train, p_hist, ax)

Above, the contour plot shows the $cost(w,b)$ over a range of $w$ and $b$. Cost levels are represented by the rings. Overlayed, using red arrows, is the path of gradient descent. Here are some things to note:
- The path makes steady (monotonic) progress toward its goal.
- initial steps are much larger than the steps near the goal.

上面的等高线图展示了 $cost(w,b)$ 在一定范围内的 $w$ 和 $b$ 上的分布。不同的环代表不同的代价水平。叠加在图上的红色箭头表示梯度下降的路径。以下是几点值得注意的现象：
- 路径稳步（单调地）朝目标前进。
- 初始步长远大于接近目标时的步长。

**Zooming in**, we can see that final steps of gradient descent. Note the distance between steps shrinks as the gradient approaches zero.

**放大视图**后，我们可以看到梯度下降的最后几步。注意，随着梯度趋近于零，每一步之间的距离不断缩小。

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(12, 4))
plt_contour_wgrad(x_train, y_train, p_hist, ax, w_range=[180, 220, 0.5], b_range=[80, 120, 0.5],
            contours=[1,5,10,20],resolution=0.5)

<a name="toc_40291_2.7.1"></a>
### Increased Learning Rate

<figure>
 <img align="left", src="../work/images/C1_W1_Lab03_alpha_too_big.PNG"   style="width:340px;height:240px;" >
</figure>
In the lecture, there was a discussion related to the proper value of the learning rate, $\alpha$ in equation(3). The larger $\alpha$ is, the faster gradient descent will converge to a solution. But, if it is too large, gradient descent will diverge. Above you have an example of a solution which converges nicely.

Let's try increasing the value of  $\alpha$ and see what happens:

### 增大学习率

课程中讨论了公式 (3) 中学习率 $\alpha$ 的合适取值。$\alpha$ 越大，梯度下降收敛到解的速度越快；但如果 $\alpha$ 过大，梯度下降就会发散（diverge）。上面展示的是一个良好收敛的示例。

让我们尝试增大 $\alpha$ 的值，看看会发生什么：

In [ ]:
# initialize parameters
w_init = 0
b_init = 0
# set alpha to a large value
iterations = 10
tmp_alpha = 8.0e-1
# run gradient descent
w_final, b_final, J_hist, p_hist = gradient_descent(x_train ,y_train, w_init, b_init, tmp_alpha, 
                                                    iterations, compute_cost, compute_gradient)

Above, $w$ and $b$ are bouncing back and forth between positive and negative with the absolute value increasing with each iteration. Further, each iteration $\frac{\partial J(w,b)}{\partial w}$ changes sign and cost is increasing rather than decreasing. This is a clear sign that the *learning rate is too large* and the solution is diverging. 
Let's visualize this with a plot.

在上面的输出中，$w$ 和 $b$ 在正负值之间来回震荡，且每次迭代绝对值都在增大。此外，每次迭代 $\frac{\partial J(w,b)}{\partial w}$ 都会改变符号，且代价不降反升。这是*学习率过大*导致解发散的明显信号。
让我们用图表来可视化这一现象。

In [ ]:
plt_divergence(p_hist, J_hist,x_train, y_train)
plt.show()

Above, the left graph shows $w$'s progression over the first few steps of gradient descent. $w$ oscillates from positive to negative and cost grows rapidly. Gradient Descent is operating on both $w$ and $b$ simultaneously, so one needs the 3-D plot on the right for the complete picture.

上面左图展示了梯度下降前几步中 $w$ 的变化过程。$w$ 在正负之间剧烈震荡，代价迅速增长。由于梯度下降同时作用于 $w$ 和 $b$，因此需要结合右侧的 3D 图来获得完整的视角。


## Congratulations!
In this lab you:
- delved into the details of gradient descent for a single variable.
- developed a routine to compute the gradient
- visualized what the gradient is
- completed a gradient descent routine
- utilized gradient descent to find parameters
- examined the impact of sizing the learning rate

## 恭喜！
在本实验中，你：
- 深入理解了单变量梯度下降的细节
- 编写了计算梯度的函数
- 可视化了梯度的含义
- 完成了梯度下降函数
- 使用梯度下降找到了最优参数
- 考察了学习率大小对算法的影响